<a href="https://colab.research.google.com/github/cameronliddle/ThesisAIDetection/blob/main/Efficientnet_b7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp -r /content/drive/MyDrive/Thesis/Processed_Dataset/resplit_dataset /content/

Mounted at /content/drive


In [ ]:

import timm
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt
import json
from tqdm import tqdm

# setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset paths
base_path = '/content/resplit_dataset'
train_dir = f"{base_path}/train"
val_dir = f"{base_path}/val"
test_dir = f"{base_path}/test"

save_dir = '/content/drive/MyDrive/Thesis'

# Load EfficientNet-B7 with pretrained weights
effnet_b7 = timm.create_model('efficientnet_b7', pretrained=False, num_classes=1)  # Pretrained model
effnet_b7 = effnet_b7.to(device)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss()

# Increase learning rate to speed up convergence
optimizer = optim.Adam(effnet_b7.parameters(), lr=5e-4)

# Increase batch size for faster training (if memory allows)
batch_size = 16  # Increase to 16 or 32 depending on GPU memory

# Transforms
img_size = (224, 224)

train_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_test_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# Datasets
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transform)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ------------------- FUNCTIONS -------------------
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, labels in tqdm(loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        preds = (torch.sigmoid(outputs) > 0.5).int()
        correct += (preds == labels.int()).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    return total_loss / len(loader), accuracy

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_labels, all_outputs = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            all_labels.extend(labels.cpu().numpy())
            all_outputs.extend(outputs.cpu().numpy())
            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)
            total_loss += loss.item()
    accuracy = correct / total
    f1 = f1_score(all_labels, (np.array(all_outputs) > 0.0).astype(int))
    precision = precision_score(all_labels, (np.array(all_outputs) > 0.0).astype(int))
    recall = recall_score(all_labels, (np.array(all_outputs) > 0.0).astype(int))
    return total_loss / len(loader), accuracy, f1, precision, recall, all_labels, all_outputs

# ------------------- TRAINING LOOP -------------------
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

epochs = 25
for epoch in range(epochs):
    train_loss, train_accuracy = train(effnet_b7, train_loader, optimizer, criterion)
    val_loss, val_accuracy, val_f1, val_precision, val_recall, _, _ = evaluate(effnet_b7, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{epochs} => "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy*100:.2f}% | "
          f"F1: {val_f1:.4f}")

test_loss, test_accuracy, test_f1, test_precision, test_recall, test_labels, test_outputs = evaluate(effnet_b7, test_loader, criterion)


# Save model
torch.save(effnet_b7.state_dict(), f"{save_dir}/efficientnet_b7.pth")

# Save training history
training_history = {
    'train_loss': train_losses,
    'val_loss': val_losses,
    'train_accuracy': [acc * 100 for acc in train_accuracies],
    'val_accuracy': [acc * 100 for acc in val_accuracies]
}
with open(f"{save_dir}/efficientnet_b7_training_history.json", 'w') as f:
    json.dump(training_history, f, indent=4)

final_results = {
    'Validation Accuracy (%)': round(val_accuracies[-1] * 100, 2),
    'Validation Loss': round(val_losses[-1], 4),
    'Validation F1-Score': round(val_f1, 4),
    'Validation Precision': round(val_precision, 4),
    'Validation Recall': round(val_recall, 4),
    'Test Accuracy (%)': round(test_accuracy * 100, 2),
    'Test Loss': round(test_loss, 4),
    'Test F1-Score': round(test_f1, 4),
    'Test Precision': round(test_precision, 4),
    'Test Recall': round(test_recall, 4)
}

with open(f"{save_dir}/efficientnet_b7_results.json", 'w') as f:
    json.dump(final_results, f, indent=4)

# Loss curve
plt.figure()
plt.plot(range(1, epochs+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('EfficientNet-B7 Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig(f"{save_dir}/efficientnet_b7_loss_curve.png")
plt.close()

# Accuracy curve
plt.figure()
plt.plot(range(1, epochs+1), [acc * 100 for acc in train_accuracies], label='Train Accuracy', marker='o', color='blue')
plt.plot(range(1, epochs+1), [acc * 100 for acc in val_accuracies], label='Validation Accuracy', marker='o', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('EfficientNet-B7 Accuracy Curve')
plt.legend()
plt.grid(True)
plt.savefig(f"{save_dir}/efficientnet_b7_accuracy_curve.png")
plt.close()

# Confusion matrix
test_preds = (np.array(test_outputs) > 0.0).astype(int)
conf_matrix = confusion_matrix(test_labels, test_preds)
cmd = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=["Fake", "Real"])
cmd.plot(cmap='Blues')
plt.title('EfficientNet-B7 Confusion Matrix')
plt.savefig(f"{save_dir}/efficientnet_b7_confusion_matrix.png")
plt.close()

# ROC Curve
fpr, tpr, _ = roc_curve(test_labels, np.array(test_outputs))
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('EfficientNet-B7 ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(f"{save_dir}/efficientnet_b7_roc_curve.png")
plt.close()

print(" All efficientnet_b7 files saved successfully!")


Training: 100%|██████████| 788/788 [02:06<00:00,  6.23it/s]


Epoch 1/25 => Train Loss: 0.9721 | Train Acc: 68.85% | Val Loss: 0.7353 | Val Acc: 71.71% | F1: 0.6592


Training: 100%|██████████| 788/788 [02:06<00:00,  6.25it/s]


Epoch 2/25 => Train Loss: 0.5412 | Train Acc: 75.60% | Val Loss: 3.0092 | Val Acc: 79.01% | F1: 0.8179


Training: 100%|██████████| 788/788 [02:06<00:00,  6.21it/s]


...

Epoch 23/25 => Train Loss: 0.1949 | Train Acc: 92.20% | Val Loss: 0.1668 | Val Acc: 93.47% | F1: 0.9359


Training: 100%|██████████| 788/788 [02:06<00:00,  6.23it/s]


Epoch 24/25 => Train Loss: 0.1765 | Train Acc: 93.25% | Val Loss: 0.1594 | Val Acc: 93.90% | F1: 0.9407


Training: 100%|██████████| 788/788 [02:06<00:00,  6.25it/s]


Epoch 25/25 => Train Loss: 0.1585 | Train Acc: 93.92% | Val Loss: 0.2120 | Val Acc: 91.70% | F1: 0.9138
✅ All efficientnet_b7 files saved successfully!
